In [0]:
ext_loc='abfss://stagingcontainer@venu72sa.dfs.core.windows.net'
csv_file=ext_loc+'/bronze/csv/Orders.csv'
json_file=ext_loc+'/bronze/json/Listings.json'  

In [0]:
dbutils.fs.ls(ext_loc)

In [0]:
dbutils.fs.mkdirs(ext_loc+'/bronze')

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
cust_schema=StructType([
    StructField('order_id', StringType(), True),
     StructField('customer_id', StringType(), True), 
     StructField('store_id', StringType(), True), 
     StructField('product_id', StringType(), True), 
     StructField('product_name', StringType(), True), 
     StructField('category', StringType(), True), 
     StructField('quantity', IntegerType(), True), 
     StructField('unit_price', DoubleType(), True), 
     StructField('total_amount', DoubleType(), True), 
     StructField('payment_type', StringType(), True), 
     StructField('order_date', TimestampType(), True)
     ])


Get the csv data

In [0]:
df = spark.read.format('csv') \
  .option('header', True) \
  .option('delimiter', ',') \
  .schema(cust_schema).load(csv_file)

df.display()

select only required columns

In [0]:
df_select=df.select("order_id","customer_id","store_id","product_name","quantity","unit_price","total_amount")
df_select.display()

In [0]:
cols = ["order_id", "customer_id","product_name","quantity","unit_price","total_amount"]
df_selctlst=df.select(*cols)
df_selctlst.display()



In [0]:
from pyspark.sql.functions import col

df_col=df.select(
    col("order_id"),
    col("customer_id"),
    col("store_id")
)
df_col.display()

Check quanty or unit_price null

In [0]:
df_clean =df_selctlst.filter("quantity IS NULL OR unit_price IS NULL or total_amount IS NULL or product_name IS NULL").show()  

In [0]:
df_cleans = df.na.drop(subset=["quantity", "total_amount", "unit_price"])
df_cleans.display()

Delete null data

In [0]:
df_clean = df.dropna(subset=["quantity","total_amount","unit_price"])
df_clean.display()


Remove duplicates

In [0]:
df_selctlst.dropDuplicates(["order_id"])

In [0]:
print(f"Before: {df.count()}")
print("After:", df_clean.count())

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import abs,when,lit
tolerance = 0.001
valid_qty = (col("quantity").isNotNull()) & (col("quantity") > 0)
valid_price = (col("unit_price").isNotNull()) & (col("unit_price") > 0)
valid_amount = (col("total_amount").isNotNull()) & (col("total_amount") > 0)
valid_total = abs(col("total_amount") - col("unit_price") * col("quantity") ) < tolerance
valid_condition=(valid_qty & valid_price & valid_amount & valid_total)
df_with_reason = df.withColumn(
    "is_valid",
    valid_condition
).withColumn(
    "error_reason",
     when(~valid_qty, lit("Invalid quantity"))
    .when(~valid_price, lit("Invalid price"))
    .when(~valid_amount, lit("Invalid total amount"))
    .when(~valid_total, lit("Mismatch total"))
    .otherwise(lit("Valid"))
)

valid_df = df_with_reason.filter(valid_condition)
invalid_df = df_with_reason.filter(~valid_condition)
invalid_df.display()
valid_df.display()

In [0]:
from pyspark.sql.functions import col, abs, when, concat_ws, coalesce, lit

# ----------------------------
# Type casting
# ----------------------------
df = df.withColumn("quantity", col("quantity")) \
       .withColumn("unit_price", col("unit_price")) \
       .withColumn("total_amount", col("total_amount"))

# ----------------------------
# reusable function
# ----------------------------
def is_positive(column_name):
    return coalesce(col(column_name) > 0, lit(False))

# ----------------------------
# validations using function
# ----------------------------
valid_qty = is_positive("quantity")

valid_price = is_positive("unit_price")

valid_amount = is_positive("total_amount")

valid_total = coalesce(
    abs(col("total_amount") - col("unit_price") * col("quantity")) < 0.01,
    lit(False)
)

# ----------------------------
# combined condition
# ----------------------------
valid_condition = valid_qty & valid_price & valid_amount & valid_total

# ----------------------------
# add validation columns
# ----------------------------
df_validated = df.withColumn(
    "is_valid",
    valid_condition
).withColumn(
    "error_reason",
    concat_ws(", ",
        when(~valid_qty, "Invalid quantity"),
        when(~valid_price, "Invalid unit price"),
        when(~valid_amount, "Invalid total amount"),
        when(~valid_total, "Total mismatch")
    )
)

# ----------------------------
# split datasets
# ----------------------------
valid_df = df_validated.filter(col("is_valid"))
invalid_df = df_validated.filter(~col("is_valid"))

invalid_df.display()
valid_df.display()

In [0]:
from pyspark.sql.functions import col, abs, when

valid_qty = (col("quantity").isNotNull()) & (col("quantity") > 0)

valid_price = (col("unit_price").isNotNull()) & (col("unit_price") > 0)

valid_amount = (col("total_amount").isNotNull()) & (col("total_amount") > 0)

valid_total = abs(
    col("total_amount") - col("unit_price") * col("quantity")
) < 0.01


# combined condition
valid_condition = valid_qty & valid_price & valid_amount & valid_total


# add validation columns
df_validated = df.withColumn(
    "is_valid",
    valid_condition
).withColumn(
    "error_reason",
    when(~valid_qty, "Invalid quantity")
    .when(~valid_price, "Invalid unit price")
    .when(~valid_amount, "Invalid total amount")
    .when(~valid_total, "Total mismatch")
)


# split datasets
valid_df = df_validated.filter(col("is_valid") == True)
invalid_df = df_validated.filter(col("is_valid") == False)


invalid_df.display()
valid_df.display()

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS vbronze")

In [0]:
valid_df.write.format("delta").mode("overwrite").saveAsTable("vbronze.bronze_orders")

In [0]:
invalid_df.write.format("delta").mode("append").saveAsTable("vbronze.bronze_orders")

In [0]:
spark.sql("select * from vbronze.bronze_orders").display()

In [0]:
new_rows = spark.createDataFrame([
    ("ORD999", "CUST10", 2)
], ["order_id", "customer_id", "quantity"])

df_updated = df.unionByName(new_rows)

In [0]:
new_row=spark.createDataFrame([("ORD999", "CUST10", 2)], ["order_id", "customer_id", "quantity"])
df_update = df.unionByName(new_row, allowMissingColumns=True)
df_update.display()